# 🔑 Solusi — Neural Network Mini dari Nol (Bab 4)> Buka HANYA setelah mencoba sendiri. Fokus pada **cara membaca kurva & angka**.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

from nn_mini.data import muat_semua
from nn_mini.layers import Dense, Sequential
from nn_mini.train import train_model

X_train, y_train, X_val, y_val, X_test, y_test = muat_semua()
print('splits:', len(y_train), len(y_val), len(y_test),
      '| positif:', int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))


## Bagian 1 — Baseline Linear

In [ ]:
linear_model = Sequential([Dense(1, activation='sigmoid', seed=0)])
hist_lin = train_model(linear_model, X_train, y_train, X_val, y_val,
                       epochs=400, batch_size=32, lr=0.5, seed=0)
print(f'Baseline linear: val acc = {hist_lin["val_acc"][-1]:.4f}')
print('-> bertahan di ±0.88-0.89 TIDAK PEDULI berapa lama dilatih: batas keputusannya garis.')


## Bagian 2 — MLP

In [ ]:
mlp = Sequential([Dense(16, activation='relu', seed=7),
                  Dense(8, activation='relu', seed=8),
                  Dense(1, activation='sigmoid', seed=9)])
hist_mlp = train_model(mlp, X_train, y_train, X_val, y_val,
                       epochs=300, batch_size=32, lr=0.1, seed=0)
print(f'MLP: val acc = {hist_mlp["val_acc"][-1]:.4f}  (val loss {hist_mlp["val_loss"][-1]:.4f})')

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_lin['train_loss'], label='linear train')
plt.plot(hist_lin['val_loss'], '--', label='linear val')
plt.plot(hist_mlp['train_loss'], label='MLP train')
plt.plot(hist_mlp['val_loss'], '--', label='MLP val')
plt.xlabel('epoch'); plt.ylabel('BCE'); plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(hist_lin['val_acc'], label='linear')
plt.plot(hist_mlp['val_acc'], label='MLP')
plt.xlabel('epoch'); plt.ylabel('val acc'); plt.legend()
plt.tight_layout(); plt.show()


**Bacaan hasil:** linear datar di 0.889 sejak epoch awal — loss turun sedikit lalustuck (konvergen ke solusi linear terbaik). MLP: loss terus turun sampai < 0.1,val acc 0.97+. Selisih ±9 poin persen = harga dari non-linearitas.

## Bagian 3 — Batas Keputusan

In [ ]:
def plot_boundary(model, judul):
    xs = np.linspace(X_train[:, 0].min() - 0.3, X_train[:, 0].max() + 0.3, 200)
    ys = np.linspace(X_train[:, 1].min() - 0.3, X_train[:, 1].max() + 0.3, 200)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    pp = model.predict_proba(grid).reshape(xx.shape)
    plt.contourf(xx, yy, pp, levels=20, cmap='coolwarm', alpha=0.6)
    plt.scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap='coolwarm', s=12, edgecolors='k', lw=0.3)
    plt.title(judul)

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1); plot_boundary(linear_model, 'Baseline linear')
plt.subplot(1, 2, 2); plot_boundary(mlp, 'MLP')
plt.tight_layout(); plt.show()


## Bagian 4 — Eksperimen Arsitektur

In [ ]:
konfigurasi = [
    ('kecil (8)',        [Dense(8, activation='relu', seed=1),  Dense(1, activation='sigmoid', seed=2)]),
    ('sedang (16-8)',    [Dense(16, activation='relu', seed=7), Dense(8, activation='relu', seed=8), Dense(1, activation='sigmoid', seed=9)]),
    ('lebar (64)',       [Dense(64, activation='relu', seed=3), Dense(1, activation='sigmoid', seed=4)]),
    ('dalam (16-16-8)',  [Dense(16, activation='relu', seed=5), Dense(16, activation='relu', seed=6), Dense(8, activation='relu', seed=10), Dense(1, activation='sigmoid', seed=11)]),
]
for nama, layers in konfigurasi:
    model = Sequential(layers)
    h = train_model(model, X_train, y_train, X_val, y_val,
                    epochs=250, batch_size=32, lr=0.1, seed=0)
    print(f'{nama:>16}: val acc = {h["val_acc"][-1]:.4f}  val loss = {h["val_loss"][-1]:.4f}')
print('-> Semua konfigurasi non-linear biasanya 0.96+. Dataset kecil: kapasitas ekstra tidak membantu.')


## Bagian 5 — Learning Rate & Early Stopping

In [ ]:
mlp_bad = Sequential([Dense(16, activation='relu', seed=7),
                      Dense(8, activation='relu', seed=8),
                      Dense(1, activation='sigmoid', seed=9)])
hist_bad = train_model(mlp_bad, X_train, y_train, X_val, y_val,
                       epochs=150, batch_size=32, lr=50.0, seed=0)
plt.plot(hist_bad['train_loss'])
plt.title('lr=50 — loss osilasi/menjauh, tidak konvergen'); plt.xlabel('epoch'); plt.show()
print(f'lr=50: val loss akhir = {hist_bad["val_loss"][-1]:.4f}  (bandingkan lr=0.1: < 0.15)')

mlp_es = Sequential([Dense(16, activation='relu', seed=7),
                     Dense(8, activation='relu', seed=8),
                     Dense(1, activation='sigmoid', seed=9)])
hist_es = train_model(mlp_es, X_train, y_train, X_val, y_val,
                      epochs=1000, batch_size=32, lr=0.1, patience=30, seed=0)
print(f'early stopping: berhenti setelah {len(hist_es["val_loss"])} epoch (dari 1000)')
print(f'val loss akhir {hist_es["val_loss"][-1]:.4f} ≈ versi 300 epoch — hemat waktu, hasil setara')


## Bagian 6 — Test Set, Sekali

In [ ]:
p_test_lin = linear_model.predict_proba(X_test)
p_test_mlp = mlp.predict_proba(X_test)
print(f'test acc baseline linear : {((p_test_lin >= 0.5) == y_test).mean():.4f}')
print(f'test acc MLP             : {((p_test_mlp >= 0.5) == y_test).mean():.4f}')


## 📝 Contoh Jawaban Analisis (ringkas)1. Batas keputusan logistic regression adalah garis (hyperplane) di ruang fitur.   Dua bulan yang mengait TIDAK bisa dipisah garis berapa pun lama training —   loss konvergen ke solusi linear terbaik (val acc ±0.889).2. Error di output diubah jadi gradien loss; tiap layer memakai cache forward-nya   (X, z) untuk menghitung seberapa besar tiap bobot berkontribusi (chain rule),   lalu meneruskan gradien ke layer sebelumnya lewat Wᵀ — begitu seterusnya sampai input.3. lr=50: langkah update jauh melebihi lebar lembah loss → parameter melompati   minimum bolak-balik → loss osilasi/menaik, tidak konvergen.4. Train loss selalu bisa diturunkan (hafalan); val loss menunjukkan generalisasi.   Patience terlalu kecil = berhenti saat val loss sedang 'napas' — kena noise epoch.5. Bobot semua nol → semua neuron di layer yang sama dapat gradien IDENTIK →   update identik → tetap identik selamanya (simetri tidak pecah) →   efektifnya cuma 1 neuron per layer. Karena itu init acak (He/Xavier).